In [5]:
# 导入所有必要的库
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig
import py3Dmol
import os

def generate_pharmacophore_visualization(smiles_string, output_filename="pharmacophore_3d_view.html"):
    """
    识别配体的药效团特征，并生成一个包含3D交互式视图的HTML文件。

    参数:
        smiles_string (str): 配体的SMILES化学式。
        output_filename (str): 输出的HTML文件名。
    """
    print(f"正在处理SMILES: {smiles_string}")

    # --- 步骤 1: 分子准备 ---
    # 从SMILES创建RDKit分子对象
    mol = Chem.MolFromSmiles(smiles_string)
    if not mol:
        print(f"错误: 无法从SMILES '{smiles_string}' 创建分子。")
        return

    # 为分子添加氢原子（这对于正确识别供体/受体至关重要）
    mol_with_hs = Chem.AddHs(mol)

    # 为分子生成一个3D构象
    # ETKDG是一种先进的构象生成算法
    status = AllChem.EmbedMolecule(mol_with_hs, AllChem.ETKDGv3())
    if status == -1:
        print("警告: 无法生成3D构象。尝试使用随机坐标。")
        AllChem.EmbedMolecule(mol_with_hs, useRandomCoords=True)
    
    # 优化构象
    AllChem.MMFFOptimizeMolecule(mol_with_hs)

    # --- 步骤 2: 药效团特征识别 ---
    # 定位RDKit内置的特征定义文件 (BaseFeatures.fdef)
    fdef_name = os.path.join(RDConfig.RDDataDir, 'BaseFeatures.fdef')
    # 构建特征工厂
    factory = ChemicalFeatures.BuildFeatureFactory(fdef_name)

    # 从分子中提取特征
    features = factory.GetFeaturesForMol(mol_with_hs)
    print(f"在分子中找到了 {len(features)} 个药效团特征。")

    # --- 步骤 3: 使用 py3Dmol 进行3D可视化 ---
    # 创建一个视图窗口
    view = py3Dmol.view(width=800, height=600)

    # 将分子（以MolBlock格式）添加到视图中
    mol_block = Chem.MolToMolBlock(mol_with_hs, confId=0)
    view.addModel(mol_block, 'mol')
    # 设置分子的显示样式为经典的“棍状”模型
    view.setStyle({'stick': {}})

    # --- 已修复 ---
    # 为不同类型的药效团特征定义颜色 (直接使用颜色名称)
    feature_colors = {
        'Donor': 'blue',        # 氢键供体: 蓝色
        'Acceptor': 'red',      # 氢键受体: 红色
        'Aromatic': 'orange',   # 芳香环: 橙色
        'Hydrophobe': 'green',  # 疏水基团: 绿色
        'PosIonizable': 'cyan', # 可正离子化: 青色
        'NegIonizable': 'magenta',# 可负离子化: 洋红色
        'ZnBinder': 'gray'      # 锌离子结合位点: 灰色
    }

    # 遍历所有找到的特征，并在其位置上添加一个彩色的球体
    for i, feat in enumerate(features):
        pos = feat.GetPos()
        family = feat.GetFamily()
        # --- 已修复 ---
        # 直接从字典获取颜色名称
        color = feature_colors.get(family, 'gray') 
        
        # 在视图中添加一个球体来代表药效团特征点
        view.addSphere({
            'center': {'x': pos.x, 'y': pos.y, 'z': pos.z},
            'radius': 0.6,
            'color': color, # 直接使用颜色名称
            'alpha': 0.8, # 设置透明度
            'label': f'{family}_{i+1}' # 为每个特征添加标签
        })
    
    
    # 自动缩放视图以适应所有内容
    view.zoomTo()
    view.show()  # 内嵌显示视图

    # --- 步骤 4: 生成并保存HTML文件 ---
    # 生成包含视图的完整HTML内容
    html_content = view._make_html()

    # 将内容写入文件
    with open(output_filename, 'w') as f:
        f.write(html_content)
    
    print(f"\n成功！可视化结果已保存到: {output_filename}")
    print("请在您的网页浏览器中打开此文件进行查看。")


# --- 主程序入口 ---
if __name__ == "__main__":
    # 示例：使用著名的胃药“西咪替丁”(Cimetidine) 的SMILES化学式
    # 它包含了多种药效团特征，是一个很好的例子
    cimetidine_smiles = "CC1=C(SC[C@H](C)NC(=N)NC)N=C[N@H]1"
    
    # 调用主函数，生成可视化文件
    generate_pharmacophore_visualization(cimetidine_smiles)

正在处理SMILES: CC1=C(SC[C@H](C)NC(=N)NC)N=C[N@H]1
在分子中找到了 12 个药效团特征。


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


成功！可视化结果已保存到: pharmacophore_3d_view.html
请在您的网页浏览器中打开此文件进行查看。
